In [2]:
# =========================
# IMPORTS
# =========================
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from torchvision import models, transforms

# =========================
# SETTINGS
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 15
EPOCHS = 10
BATCH_SIZE = 24
FOLDS = 5

TRAIN_DIR = "/kaggle/input/dl-lab-1-image-classification/train/train"
TEST_DIR  = "/kaggle/input/dl-lab-1-image-classification/test_images/test_images"
TEST_CSV  = "/kaggle/input/dl-lab-1-image-classification/test.csv"

# =========================
# EMA
# =========================
class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] +
                    (1 - self.decay) * param.data
                )

    def apply(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.shadow[name]

# =========================
# DATA PREPARATION
# =========================
classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {c:i for i,c in enumerate(classes)}

samples = []
for cls in classes:
    path = os.path.join(TRAIN_DIR, cls)
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith((".jpg",".png",".jpeg")):
                samples.append((os.path.join(root,f), class_to_idx[cls]))

paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

# =========================
# TRANSFORMS
# =========================
train_tf = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.RandomResizedCrop(300, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# =========================
# DATASET
# =========================
class CustomDataset(Dataset):
    def __init__(self, paths, labels=None, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, self.labels[idx]
        return img

# =========================
# TRAIN 5-FOLD
# =========================
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

model_files = []

for fold,(train_idx,val_idx) in enumerate(skf.split(paths,labels)):

    print(f"\n🔥 FOLD {fold}")

    train_ds = CustomDataset([paths[i] for i in train_idx],
                             [labels[i] for i in train_idx],
                             train_tf)

    train_loader = DataLoader(train_ds,
                              batch_size=BATCH_SIZE,
                              shuffle=True)

    model = models.efficientnet_b3(
        weights="IMAGENET1K_V1"
    )
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES
    )
    model = model.to(device)

    ema = EMA(model)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(),
                            lr=2e-4,
                            weight_decay=1e-4)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS)

    for epoch in range(EPOCHS):

        model.train()

        for images, targets in tqdm(train_loader):
            images, targets = images.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            ema.update()

        scheduler.step()

    torch.save(model.state_dict(), f"model_fold_{fold}.pth")
    model_files.append(f"model_fold_{fold}.pth")

# =========================
# INFERENCE + STRONG TTA
# =========================

print("\n🚀 Creating submission...")

test_df = pd.read_csv(TEST_CSV)

base_tf = val_tf

models_list = []

for file in model_files:
    model = models.efficientnet_b3(weights=None)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES
    )
    model.load_state_dict(torch.load(file))
    model.to(device)
    model.eval()
    models_list.append(model)

preds = []

with torch.no_grad():
    for img_name in tqdm(test_df.image_id):

        path = os.path.join(TEST_DIR, img_name)
        img = Image.open(path).convert("RGB")

        imgs = []

        imgs.append(base_tf(img))
        imgs.append(base_tf(transforms.functional.hflip(img)))

        resized = transforms.functional.resize(img,(300,300))
        imgs.append(base_tf(resized))

        imgs.append(base_tf(transforms.functional.hflip(resized)))

        imgs = torch.stack(imgs).to(device)

        total = None

        for model in models_list:
            out = model(imgs)
            prob = torch.softmax(out, dim=1)
            prob = prob.mean(dim=0)

            if total is None:
                total = prob
            else:
                total += prob

        total /= len(models_list)
        preds.append(torch.argmax(total).item())

submission = pd.DataFrame({
    "image_id": test_df.image_id,
    "label": preds
})

submission.to_csv("submission.csv", index=False)

print("🔥 DONE. submission.csv готов")


🔥 FOLD 0
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 193MB/s] 
100%|██████████| 330/330 [03:47<00:00,  1.45it/s]



🔥 FOLD 1


100%|██████████| 330/330 [03:47<00:00,  1.45it/s]



🔥 FOLD 2


...


🔥 FOLD 4


100%|██████████| 330/330 [03:46<00:00,  1.46it/s]



🚀 Creating submission...


100%|██████████| 2503/2503 [04:44<00:00,  8.80it/s]

🔥 DONE. submission.csv готов


In [3]:
import os

print("Files in working directory:")
print(os.listdir("/kaggle/working/"))

Files in working directory:
['model_fold_4.pth', 'model_fold_1.pth', 'model_fold_2.pth', '.virtual_documents', 'model_fold_3.pth', 'model_fold_0.pth', 'submission.csv']
